Exploratory EDA to inform truck trip generation equations modeling effort

In [ ]:
import pandas as pd
import numpy as np 
import geopandas as gpd

import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import pearsonr
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV

In [ ]:
targets = pd.read_csv("../data/processed/truck_trip_generation_zone.csv", index_col = "TAZ1454")
targets.index.name = "taz_id"
features = pd.read_csv("../data/external/mtc/2023_TM161_IPA_35/landuse/tazData.csv", index_col = "ZONE")
features.index.name = "taz_id"
gdf = gpd.read_file("../data/external/mtc/MTC_TAZ/Travel Analysis Zones.shp")
gdf = gdf.set_index("TAZ1454")
gdf.index.name = "taz_id"

In [ ]:
cols_map = {
    "HT_NF_P" : [c for c in targets.columns if c.startswith('HT_NF') and c.endswith('production')], 
    "HT_NF_A" : [c for c in targets.columns if c.startswith('HT_NF') and c.endswith('attraction')],  
    "MD_NF_P" : [c for c in targets.columns if (c.startswith('MT1_NF') or c.startswith('MT2_NF') )and c.endswith('production')],  
    "MD_NF_A" : [c for c in targets.columns if (c.startswith('MT1_NF') or c.startswith('MT2_NF') )and c.endswith('attraction')],   
    "LT_NF_P" : [c for c in targets.columns if c.startswith('LT_NF') and c.endswith('production')],   
    "LT_NF_A" : [c for c in targets.columns if c.startswith('LT_NF') and c.endswith('attraction')],    
    "HT_FR_P" : [c for c in targets.columns if c.startswith('HT_FR') and c.endswith('production')], 
    "HT_FR_A" : [c for c in targets.columns if c.startswith('HT_FR') and c.endswith('attraction')],  
    "MD_FR_P" : [c for c in targets.columns if (c.startswith('MT1_FR') or c.startswith('MT2_FR') )and c.endswith('production')],  
    "MD_FR_A" : [c for c in targets.columns if (c.startswith('MT1_FR') or c.startswith('MT2_FR') )and c.endswith('attraction')],  
    "LT_FR_P" : [c for c in targets.columns if c.startswith('LT_FR') and c.endswith('production')],  
    "LT_FR_A" : [c for c in targets.columns if c.startswith('LT_FR') and c.endswith('attraction')],  
}

target_cols = []
for name, cols in cols_map.items():
    targets[name] = targets[cols].sum(axis = 1) 
    # targets[f"log_{name}"] = np.log1p(targets[cols].sum(axis = 1))
    target_cols.append(name)
    # target_cols.append(f"log_{name}")

In [ ]:
features["emp_density"] = features["TOTEMP"]/features["TOTACRE"]
features["pop_density"] = features["TOTHH"]/features["TOTACRE"]
features["log_emp"] = np.log1p(features["TOTEMP"])
features["sqrt_TOTEMP"] = np.sqrt(features["TOTEMP"])
features["share_employment"] = features["TOTEMP"]/features["TOTEMP"].sum()

# Composition
features["CMP_RETEMPN"] = features["RETEMPN"]/features["TOTEMP"]
features["CMP_FPSEMPN"] = features["FPSEMPN"]/features["TOTEMP"]
features["CMP_HEREMPN"] = features["HEREMPN"]/features["TOTEMP"]
features["CMP_AGREMPN"] = features["AGREMPN"]/features["TOTEMP"]
features["CMP_MWTEMPN"] = features["MWTEMPN"]/features["TOTEMP"]
features["CMP_OTHEMPN"] = features["OTHEMPN"]/features["TOTEMP"]

# Intensity
features["INT_RETEMPN_TOT"] = features["RETEMPN"]/features["TOTACRE"]
features["INT_FPSEMPN_TOT"] = features["FPSEMPN"]/features["TOTACRE"]
features["INT_HEREMPN_TOT"] = features["HEREMPN"]/features["TOTACRE"]
features["INT_AGREMPN_TOT"] = features["AGREMPN"]/features["TOTACRE"]
features["INT_MWTEMPN_TOT"] = features["MWTEMPN"]/features["TOTACRE"]
features["INT_OTHEMPN_TOT"] = features["OTHEMPN"]/features["TOTACRE"]
features["INT_RETEMPN_CI"] = features["RETEMPN"]/features["CIACRE"]
features["INT_FPSEMPN_CI"] = features["FPSEMPN"]/features["CIACRE"]
features["INT_HEREMPN_CI"] = features["HEREMPN"]/features["CIACRE"]
features["INT_AGREMPN_CI"] = features["AGREMPN"]/features["CIACRE"]
features["INT_MWTEMPN_CI"] = features["MWTEMPN"]/features["CIACRE"]
features["INT_OTHEMPN_CI"] = features["OTHEMPN"]/features["CIACRE"]

# Interactins
features["RETEMPN_CIACRE"] = np.log1p(features["RETEMPN"]) * features["CIACRE"]
features["FPSEMPN_CIACRE"] = np.log1p(features["FPSEMPN"]) * features["CIACRE"]
features["HEREMPN_CIACRE"] = np.log1p(features["HEREMPN"]) * features["CIACRE"]
features["AGREMPN_CIACRE"] = np.log1p(features["AGREMPN"]) * features["CIACRE"]
features["MWTEMPN_CIACRE"] = np.log1p(features["MWTEMPN"]) * features["CIACRE"]
features["OTHEMPN_CIACRE"] = np.log1p(features["OTHEMPN"]) * features["CIACRE"]

In [ ]:
# geomap = gdf.merge(targets[target_cols], how = "left", left_index = True, right_index = True).merge(features, how = "left", left_index = True, right_index = True)
# geomap.to_file("../data/interim/geo/taz1454_truck_trips_land_use.shp")

In [ ]:
# 1. Combine them side-by-side (rows align perfectly because your indexes match)
feature_cols = [
    # 'DISTRICT', 
    # 'SD', 
    # 'COUNTY', 
    # Population 
    'TOTHH', 
    'HHPOP',
    'TOTPOP',
    'HHINCQ1',
    'HHINCQ2',
    'HHINCQ3',
    'HHINCQ4',
    'SHPOP62P', 
    'AGE0004', 
    'AGE0519',
    'AGE2044',
    'AGE4564',
    'AGE65P', 
    'HSENROLL', 
    'COLLFTE', 
    'COLLPTE',
    "pop_density",
    
    #Employment 
    # "share_employment",
    'TOTEMP',
    'sqrt_TOTEMP',
    'RETEMPN',
    'FPSEMPN',
    'HEREMPN',
    'AGREMPN', 
    'MWTEMPN',
    'OTHEMPN', 
    # 'EMPRES',
    "emp_density",  'CMP_RETEMPN',
       'CMP_FPSEMPN', 'CMP_HEREMPN', 'CMP_AGREMPN', 'CMP_MWTEMPN',
       'CMP_OTHEMPN', 'INT_RETEMPN_TOT', 'INT_FPSEMPN_TOT', 'INT_HEREMPN_TOT',
       'INT_AGREMPN_TOT', 'INT_MWTEMPN_TOT', 'INT_OTHEMPN_TOT',
       'INT_RETEMPN_CI', 'INT_FPSEMPN_CI', 'INT_HEREMPN_CI', 'INT_AGREMPN_CI',
       'INT_MWTEMPN_CI', 'INT_OTHEMPN_CI',
    'RETEMPN_CIACRE', 'FPSEMPN_CIACRE',
       'HEREMPN_CIACRE', 'AGREMPN_CIACRE', 'MWTEMPN_CIACRE', 'OTHEMPN_CIACRE',
    
    # Land Use 
    'TOTACRE',
    'RESACRE', 
    'CIACRE',
    # 'PRKCST', 
    # 'OPRKCST', 
    # 'AREATYPE',
    # 'TERMINAL',
    # 'SFDU',
    # 'MFDU', 
    # 'TOPOLOGY',
]

combined_df = pd.concat([features, targets[target_cols]], axis=1)
full_corr = combined_df.corr()
corr_matrix = full_corr.loc[feature_cols, target_cols]
top_5_results = {}

for target in target_cols:
    target_series = corr_matrix[target]
    top_5_features = target_series.abs().nlargest(5).index
    top_5_results[target] = [
        f"{target_series[feat]:.4f} (abs: {abs(target_series[feat]):.4f}) via {feat}"
        for feat in top_5_features
    ]

summary_df = pd.DataFrame(top_5_results)
summary_df.index = [f"Rank {i+1}" for i in range(5)]
summary_df.T

In [ ]:
combined_df.to_csv("data_for_trip_generation.csv")

In [ ]:
vars_list = feature_cols  # your list of variables
y_var = "HT_FR_P"
X = combined_df[feature_cols]
y = combined_df[y_var]

In [ ]:
lasso = make_pipeline(
    StandardScaler(),
    LassoCV(cv=5)
)

lasso.fit(X, y)

coefs = lasso.named_steps["lassocv"].coef_

variables = []
coefficients = []
for var, coef in zip(X.columns, coefs):
    if coef > 0: 
        variables.append(var)
        coefficients.append(coef)
        print(var, coef)

In [ ]:
n = len(feature_cols)
fig, axes = plt.subplots(n, 2, figsize=(12, 4*n))

for i, x_var in enumerate(feature_cols):
    df = combined_df[[x_var, y_var]]
    r, _ = pearsonr(df[x_var], df[y_var])
    r_log, _ = pearsonr(np.log1p(df[x_var]), df[y_var])

    # --- LEFT: linear plot ---
    ax1 = axes[i, 0] if n > 1 else axes[0]
    sns.scatterplot(data=df, x=x_var, y=y_var, ax=ax1)
    sns.regplot(data=df, x=x_var, y=y_var, ax=ax1, scatter=False)
    ax1.set_title(f"{y_var} (linear)\nr = {r:.2f}")

    # --- RIGHT: log-log plot ---
    ax2 = axes[i, 1] if n > 1 else axes[1]
    sns.scatterplot(data=df, x=x_var, y=y_var, ax=ax2)
    sns.regplot(data=df, x=x_var, y=y_var, ax=ax2, scatter=False)
    ax2.set_xscale("log")
    ax2.set_title(f"{y_var} (log)\nr = {r_log:.2f}")

plt.tight_layout()
plt.show()